# 我使用了什么库？

In [ ]:
# 需要导入的库

import numpy as np
import datetime

import torch
import torch.optim as optim
import torch.nn as nn
import torch.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from torch.utils.tensorboard import SummaryWriter

import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('fivethirtyeight')

# 然后，我要干什么？

打算继续改造训练循环,这次会使用类来封装这些过程
- 构造方法
- 类的限制符

然后实例化类来实现一个**管道**

# 类的设计

## 类的基本使用

**Q**: 方法和带'_'和'__'前缀的方法有什么区别

结合代码:

**Q**: `setattr`起到了什么作用？如何使用？

In [12]:
# 程序2-101: 简单的类

# 一个空的定义
class Foo(object):
    pass

# 构造方法,方法
class Foo(object):
    def __init__(self, x,n=10):
        self.x = x
        self.n = n
    def print(self):
        print(self.x ** self.n)
    def __private_print(self):
        print("This is private print!")
    def _protected_print(self):
        # 访问private成员
        self.__private_print()
        def func_print(x):
            print(x ** self.n)
        return func_print

Foo(2).print()
Foo(2,3).print()

# 访问protected成员
Foo(2,3)._protected_print()(3)


1024
8
This is private print!
27


In [13]:
# 程序2-102: setattr

class Dog(object):
    def __init__(self, name):
        self.name = name

rex = Dog('Rex')
print(rex.name)

def bark(dog):
    print('{} barks: "Woof!"'.format(dog.name))

bark(rex)

# 使用setattr
def bark_method(self):
    print('{} barks: "Woof!"'.format(self.name))

setattr(Dog,'bark',bark_method)

fido = Dog('Fido')
fido.bark()


Rex
Rex barks: "Woof!"
Fido barks: "Woof!"


## 类的设计

**Q**: 训练循环中哪些抽象出来后不会影响训练循环的逻辑？

**Q**: 除了以上，该类有哪些必选参数？

**Q**: 该类有哪些可选参数？

**Q**: 可选参数和必选参数的处理有什么不同

结合代码:

**Q**: `torch.backends.cudnn.deterministic=True`和`torch.backends.cudnn.benchmark = False`的作用，参考pytorch的可重复性指南


In [14]:
# 程序2-103: 类的设计和实现

class StepByStep(object):
    # 构造函数
    def __init__(self, model,loss_fn,optimizer):
        # 必选属性
        self.model = model
        self.loss_fn = loss_fn
        self.optimizer = optimizer
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        ## 即刻发送模型到设备
        self.model.to(self.device)
        ## 参数
        self.losses = []
        self.val_losses = []
        self.total_steps = 0
        ## 处理逻辑
        self.train_step = self._make_train_step()
        self.val_step =  self._make_val_step()

        # 可选项
        self.train_loader = None
        self.val_loader = None
        self.writer = None
    # 封装指定设备的动作
    def to(self,device):
        self.device = device
        self.model.to(self.device)
    # 可选参数的setter
    def set_loader(self,train_loader,val_loader=None):
        self.train_loader = train_loader
        self.val_loader = val_loader
    def set_tensorboard(self,name,folder='logs'):
        suffix = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        self.writer = SummaryWriter('{}/{}_{}'.format(folder,name,suffix))


In [ ]:
# 程序2-104: 使用setattr添加属性(批量梯度下降)
def _make_train_step(self):
    def train_step(x,y):
        # 设置模式
        self.model.train()

        # 前向传递,直接使用类来调用钩子
        yhat = self.model(x)
        # 计算损失
        loss = self.loss_fn(yhat,y)
        # 反向传播
        loss.backward()
        # 使用优化器更新参数
        self.optimizer.step()
        # 注意梯度清除
        self.optimizer.zero_grad()

        # 注意按cpu返回数据
        return loss.item()
    return train_step

def _make_val_step(self):
    def val_step(x,y):
        # 设置模式
        self.model.eval()

        # 前向传递,直接使用类来调用钩子
        yhat = self.model(x)
        # 计算损失
        loss = self.loss_fn(yhat,y)
        # 反向传播
        #loss.backward()
        # 使用优化器更新参数
        #self.optimizer.step()
        # 注意梯度清除
        #self.optimizer.zero_grad()

        # 注意按cpu返回数据
        return loss.item()
    return val_step

setattr(StepByStep,'_make_train_step',_make_train_step)
setattr(StepByStep,'_make_val_step',_make_val_step)

In [ ]:
# 程序2-105: 使用setattr添加属性(小批量梯度下降)
# 注意 全量梯度下降的方法其实是为了小批量提供了内循环的函数

def _mini_batch(self,validation=False):
    if validation:
        data_loader = self.val_loader
        step = self.val_step
    else:
        data_loader = self.train_loader
        step = self.train_step
    if data_loader is None:
        return None

    # 训练循环
    mini_batch_losses = []
    for x_batch, y_batch in data_loader:
        x_batch = x_batch.to(self.device)
        y_batch = y_batch.to(self.device)

        mini_batch_losses.append(step(x_batch,y_batch))
    loss = np.mean(mini_batch_losses)
    return loss

setattr(StepByStep,'_mini_batch',_mini_batch)


In [ ]:
# 程序2-106: 随机数和训练循环

# 随机数
def set_seed(self,seed=42):
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark = False
    torch.manual_seed(seed)
    np.random.seed(seed)

def train(self,n_epochs,seed=42):
    # 设置随机种子确保实验可重复
    self.total_epochs +=1

    for epoch in range(n_epochs):
        # 内循环
        # 训练
        loss = self.mini_batch(validation=False)
        self.losses.append(loss)

        # 预测
        with torch.no_grad():
            val_loss = self.val_step(validation=True)
            self.val_losses.append(val_loss)

    # 处理TensorBoard
        if self.writer:
            scalars = {'training': loss}
            if val_loss is not None:
                scalars.update({'validation': val_loss})
            # Records both losses for each epoch under the main tag "loss"
            self.writer.add_scalars(main_tag='loss',
                                    tag_scalar_dict=scalars,
                                    global_step=epoch)
    if self.writer:
        # Flushes the writer
        self.writer.flush()

setattr(StepByStep,'set_seed',set_seed)
setattr(StepByStep,'train',train)

In [ ]:
# 程序2-107: 保存和加载方法

# 保存
def save_checkpoint(self, filename):
    # 构建字典
    checkpoint = {'epoch': self.total_epochs,
                  'model_state_dict': self.model.state_dict(),
                  'optimizer_state_dict': self.optimizer.state_dict(),
                  'loss': self.losses,
                  'val_loss': self.val_losses}

    torch.save(checkpoint, filename)

setattr(StepByStep, 'save_checkpoint', save_checkpoint)

# 加载
def load_checkpoint(self, filename):
    # 加载字典，注意参数
    checkpoint = torch.load(filename, weights_only=False)

    # 解析参数
    self.model.load_state_dict(checkpoint['model_state_dict'])
    self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    self.total_epochs = checkpoint['epoch']
    self.losses = checkpoint['loss']
    self.val_losses = checkpoint['val_loss']

    self.model.train() # always use TRAIN for resuming training

setattr(StepByStep, 'load_checkpoint', load_checkpoint)

In [ ]:
# 程序2-108: 预测函数

def predict(self, x):
    # 设置模式
    self.model.eval()

    # 构造张量
    x_tensor = torch.as_tensor(x).float()

    # 预测
    y_hat_tensor = self.model(x_tensor.to(self.device))

    # 回退默认模式
    self.model.train()

    # 注意返回的是cpu张量
    return y_hat_tensor.detach().cpu().numpy()

setattr(StepByStep, 'predict', predict)

In [ ]:
# 程序2-109: 可视化（plt和处理tensboard）

# plt简易绘图

def plot_losses(self):
    fig = plt.figure(figsize=(10, 4))
    plt.plot(self.losses, label='Training Loss', c='b')
    if self.val_loader:
        plt.plot(self.val_losses, label='Validation Loss', c='r')
    plt.yscale('log')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.tight_layout()
    return fig

setattr(StepByStep, 'plot_losses', plot_losses)


# tenseboard
def add_graph(self):
    if self.train_loader and self.writer:
        # 获取计算图
        x_sample, y_sample = next(iter(self.train_loader))
        self.writer.add_graph(self.model, x_sample.to(self.device))

setattr(StepByStep, 'add_graph', add_graph)

## 数据准备

即使使用类做抽象，数据准备和模型的配置是不用变的

In [ ]:
# 程序2-106: 数据准备

# 数据准备
%run -i tmp_script/v2_data_pre.py

# 模型准备

%run -i tmp_script/v4_model_config.py

# 验证

print(model.state_dict())
